In [11]:
import pandas as pd
import os
features_folder = r"C:\Users\eduar\Projects\Python\quant-project\data\raw"

file_path = os.path.join(features_folder, "BAC.csv")



def load_and_clean_price_csv(path: str) -> pd.DataFrame:
    """
    Cleans malformed price CSVs by:
    - Finding the row where 'Date' starts
    - Dropping all rows above it
    - Enforcing a standard OHLCV schema
    """

    raw = pd.read_csv(path, header=None)

    # Find row where first column is 'Date'
    date_row_idx = raw.index[raw.iloc[:, 0] == "Date"]

    if len(date_row_idx) == 0:
        raise ValueError(f"Cannot find Date row in {path}")

    start_idx = date_row_idx[0] + 1

    # Slice actual data
    df = raw.iloc[start_idx:].copy()

    # Enforce standard column names
    expected_cols = ["Date", "Close", "High", "Low", "Open", "Volume"]

    if df.shape[1] < len(expected_cols):
        raise ValueError(
            f"{path}: Expected at least {len(expected_cols)} columns, "
            f"found {df.shape[1]}"
        )

    df = df.iloc[:, : len(expected_cols)]
    df.columns = expected_cols

    # Parse Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.set_index("Date").sort_index()

    # Convert numeric columns
    for col in ["Close", "High", "Low", "Open", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop invalid rows
    df = df.dropna(subset=["Close"])

    return df
load_and_clean_price_csv(file_path)


,Close,High,Low,Open,Volume
Date,,,,,
2024-01-02,32.310413,32.472440,31.709953,31.824325,36668600
2024-01-03,31.957754,32.186502,31.681355,32.072130,45988700
2024-01-04,32.215092,32.701179,31.967285,31.995877,39834600
2024-01-05,32.815552,33.063359,32.129312,32.215092,49242400
2024-01-08,32.558216,32.691651,32.062599,32.691651,40253900
...,...,...,...,...,...
2025-12-08,53.900002,54.209999,53.490002,53.900002,34386600
2025-12-09,53.540001,54.279999,53.270000,53.970001,47189300
2025-12-10,54.080002,54.549999,53.340000,53.590000,54424400
